In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from sklearn.preprocessing import StandardScaler
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D1 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,0,0,15.438583,22.841,263.611623,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,20,1,8.829353,5.660,36.980700,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,6,1,13.476123,7.791,74.636342,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,45,1,8.632732,7.908,46.791979,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,59,0,9.783954,15.237,107.637514,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,55,1,31.338410,6.110,144.490782,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,174,1,13.041604,7.182,69.214868,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,0,1,8.944517,16.483,102.594274,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,0,0,14.184236,9.981,103.229492,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

In [14]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [15]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 14)
y_train:  (139,)


In [16]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [17]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 16)

# Feature Selection: RENT

In [18]:
# Choose features from the result of RENT 
plsr = ["age",
"cavum_oris",
"hpv_related",
"uicc8_III-IV",
"oropharynx",
"pack_years",
"charlson"]



In [19]:
X_plsr = X.loc[:, plsr]
X_new = X_plsr.copy()

In [20]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [23]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [24]:
X_new

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,54.238356,0,0.0,0.0,1,0.000000,0
1,54.539726,0,0.0,0.0,0,27.404795,1
2,59.019178,1,0.0,1.0,0,41.019178,1
3,70.726027,0,0.0,0.0,0,37.500000,1
4,67.865753,0,0.0,0.0,0,53.000000,1
...,...,...,...,...,...,...,...
134,60.435616,0,1.0,0.0,1,0.000000,0
135,68.794521,0,1.0,1.0,1,0.000000,0
136,57.498630,0,1.0,0.0,1,39.498630,1
137,65.684932,0,1.0,1.0,1,71.527397,1


In [25]:
X_new_std

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,-0.776210,0,0.0,0.0,1,-1.101176,0
1,-0.737256,0,0.0,0.0,0,0.105775,1
2,-0.158251,1,0.0,1.0,0,0.705374,1
3,1.354953,0,0.0,0.0,0,0.550384,1
4,0.985240,0,0.0,0.0,0,1.233028,1
...,...,...,...,...,...,...,...
134,0.024835,0,1.0,0.0,1,-1.101176,0
135,1.105290,0,1.0,1.0,1,-1.101176,0
136,-0.354794,0,1.0,0.0,1,0.638406,1
137,0.703351,0,1.0,1.0,1,2.049005,1


In [26]:
MAASTRO_new 

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,55,0,1,0,1,0,1
1,55,0,0,1,1,20,0
2,55,0,0,1,1,6,1
3,61,0,0,1,0,45,1
4,70,0,1,0,1,59,1
...,...,...,...,...,...,...,...
94,66,0,0,1,0,55,0
95,63,0,0,1,0,174,1
96,63,0,1,1,1,0,1
97,54,0,1,0,1,0,0


In [27]:
MAASTRO_new_std

,age,cavum_oris,hpv_related,uicc8_III-IV,oropharynx,pack_years,charlson
0,-0.677762,0,1,0,1,-1.101176,1
1,-0.677762,0,0,1,1,-0.220344,0
2,-0.677762,0,0,1,1,-0.836927,1
3,0.097786,0,0,1,0,0.880696,1
4,1.261108,0,1,0,1,1.497278,1
...,...,...,...,...,...,...,...
94,0.744076,0,0,1,0,1.321112,0
95,0.356302,0,0,1,0,6.562062,1
96,0.356302,0,1,1,1,-1.101176,1
97,-0.807020,0,1,0,1,-1.101176,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [28]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 23:51:39,956] A new study created in memory with name: no-name-cbf25c65-c74d-4c3a-94b5-bfe740f9c4f1


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844


[I 2024-04-13 23:52:00,263] A new study created in memory with name: no-name-b1ec1ca6-bcec-4d08-bc7a-ff1d2bf91059


Fold 5 C-index: 0.6431924882629108
[I 2024-04-13 23:52:00,250] Trial 0 finished with value: 0.760896852663171 and parameters: {}. Best is trial 0 with value: 0.760896852663171.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.760896852663171], datetime_start=datetime.datetime(2024, 4, 13, 23, 51, 40, 582110), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 0, 249534), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.760896852663171


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16131595626689033
Fold 2 IBS: 0.17652730860451066
Fold 3 IBS: 0.15314029222827888
Fold 4 IBS: 0.14780969255866613
Fold 5 IBS: 0.25578610957274617
[I 2024-04-13 23:52:01,552] Trial 0 finished with value: 0.17891587184621843 and parameters: {}. Best is trial 0 with value: 0.17891587184621843.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17891587184621843], datetime_start=datetime.datetime(2024, 4, 13, 23, 52, 0, 331420), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 1, 552282), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17891587184621843


In [29]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [30]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.761
train_ibs:  0.179


#### Test

In [31]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [32]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.568
IBS score: 0.279


In [33]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [34]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:52:02,152] A new study created in memory with name: no-name-380428d4-a552-4c41-bc80-4fb3069b47ef


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.6383928571428571
Fold 3 C-index: 0.7720588235294118


[I 2024-04-13 23:52:02,726] A new study created in memory with name: no-name-5b59c2a6-76e6-4a55-84ff-31c4520fb118


Fold 4 C-index: 0.8037974683544303
Fold 5 C-index: 0.5633802816901409
[I 2024-04-13 23:52:02,718] Trial 0 finished with value: 0.691456622074104 and parameters: {}. Best is trial 0 with value: 0.691456622074104.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.691456622074104], datetime_start=datetime.datetime(2024, 4, 13, 23, 52, 2, 190450), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 2, 718249), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.691456622074104


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2139765166947049
Fold 2 IBS: 0.2215779101687221
Fold 3 IBS: 0.20453594089893293
Fold 4 IBS: 0.2247380328252742
Fold 5 IBS: 0.21812431423859643
[I 2024-04-13 23:52:03,847] Trial 0 finished with value: 0.2165905429652461 and parameters: {}. Best is trial 0 with value: 0.2165905429652461.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905429652461], datetime_start=datetime.datetime(2024, 4, 13, 23, 52, 2, 951086), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 3, 834196), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905429652461


In [36]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.691
train_ibs:  0.217


#### Test

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.534


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:52:04,501] A new study created in memory with name: no-name-62052bd2-b1b1-4d4c-bf02-586dd4bb0818


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8438818565400844


[I 2024-04-13 23:52:06,089] A new study created in memory with name: no-name-fd9f1e48-d7ee-4a16-ab41-2ed2a9489ebb


Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:06,077] Trial 0 finished with value: 0.7582867625323683 and parameters: {}. Best is trial 0 with value: 0.7582867625323683.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7582867625323683], datetime_start=datetime.datetime(2024, 4, 13, 23, 52, 4, 742774), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 6, 77116), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7582867625323683


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16153564154189293
Fold 2 IBS: 0.1842400309665049
Fold 3 IBS: 0.15275060525197218
Fold 4 IBS: 0.14653906831683322
Fold 5 IBS: 0.25395902869786563
[I 2024-04-13 23:52:07,820] Trial 0 finished with value: 0.17980487495501377 and parameters: {}. Best is trial 0 with value: 0.17980487495501377.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.17980487495501377], datetime_start=datetime.datetime(2024, 4, 13, 23, 52, 6, 130136), datetime_complete=datetime.datetime(2024, 4, 13, 23, 52, 7, 819606), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.17980487495501377


In [42]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.758
train_ibs:  0.18


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.571


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.275


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 23:52:08,865] A new study created in memory with name: no-name-148bd728-9baa-4f07-9741-da6a533446e8


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:10,629] Trial 0 finished with value: 0.7582867625323683 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7582867625323683.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:12,586] Trial 1 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7582867625323683.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:14,511] Trial 2 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:51,871] Trial 24 finished with value: 0.7582867625323683 and parameters: {'l1_ratio': 0.5910098988284204}. Best is trial 5 with value: 0.7591796196752254.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:53,639] Trial 25 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.29037524082647914}. Best is trial 5 with value: 0.7591796196752254.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:52:55,036] Trial 26 finished with value: 0.7582867625323683 and parameters: {'l1_ratio': 0.8401513352750697

Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:53:29,882] Trial 47 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.2634506000463872}. Best is trial 5 with value: 0.7591796196752254.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:53:31,464] Trial 48 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.5252656327220788}. Best is trial 5 with value: 0.7591796196752254.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6384976525821596
[I 2024-04-13 23:53:33,087] Trial 49 finished with value: 0.7581992275183627 and parameters: {'l1_ratio': 0.37543149390939745}. Best is trial 5 with value: 0.7591796196752254.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8

...

Fold 1 IBS: 0.16147755914048548
Fold 2 IBS: 0.18363855053607006
Fold 3 IBS: 0.1526190676839271
Fold 4 IBS: 0.14653952970577372
Fold 5 IBS: 0.2539331115447447
[I 2024-04-13 23:55:36,595] Trial 0 finished with value: 0.17964156372220025 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.17964156372220025.
Fold 1 IBS: 0.16137427153465642
Fold 2 IBS: 0.18267947672298618
Fold 3 IBS: 0.1524682185261851
Fold 4 IBS: 0.1464604748170855
Fold 5 IBS: 0.25375770370810097
[I 2024-04-13 23:55:40,244] Trial 1 finished with value: 0.1793480290618028 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.1793480290618028.
Fold 1 IBS: 0.16135847880812645
Fold 2 IBS: 0.1825925025988472
Fold 3 IBS: 0.15241456985698845
Fold 4 IBS: 0.14651501820866328
Fold 5 IBS: 0.2536446994907515
[I 2024-04-13 23:55:42,344] Trial 2 finished with value: 0.1793050537926754 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.1793050537926754.


Fold 1 IBS: 0.16130347009927784
Fold 2 IBS: 0.17972938075739273
Fold 3 IBS: 0.15235581923762645
Fold 4 IBS: 0.22171990965951452
Fold 5 IBS: 0.25358621012215415
[I 2024-04-13 23:56:40,466] Trial 25 finished with value: 0.19373895797519314 and parameters: {'l1_ratio': 0.03981317955666948}. Best is trial 12 with value: 0.17869979182273077.
Fold 1 IBS: 0.16134724753152271
Fold 2 IBS: 0.18250566604419238
Fold 3 IBS: 0.15239625807347582
Fold 4 IBS: 0.14651111937282857
Fold 5 IBS: 0.25361588939788765
[I 2024-04-13 23:56:42,833] Trial 26 finished with value: 0.1792752360839814 and parameters: {'l1_ratio': 0.18783928730207888}. Best is trial 12 with value: 0.17869979182273077.
Fold 1 IBS: 0.1614445278333994
Fold 2 IBS: 0.18340671073131992
Fold 3 IBS: 0.15257506097361084
Fold 4 IBS: 0.14654715180507763
Fold 5 IBS: 0.2538816754008691
[I 2024-04-13 23:56:45,256] Trial 27 finished with value: 0.17957102534885538 and parameters: {'l1_ratio': 0.58416616744615}. Best is trial 12 with value: 0.17869979

Fold 5 IBS: 0.2536793531656018
[I 2024-04-13 23:57:30,932] Trial 49 finished with value: 0.1791960641045191 and parameters: {'l1_ratio': 0.12479818361037143}. Best is trial 41 with value: 0.17867968268861573.
Fold 1 IBS: 0.2126538011301309
Fold 2 IBS: 0.1796705082688478
Fold 3 IBS: 0.15234301113818527
Fold 4 IBS: 0.2226021280057045
Fold 5 IBS: 0.25355575417313825
[I 2024-04-13 23:57:33,768] Trial 50 finished with value: 0.2041650405432013 and parameters: {'l1_ratio': 0.02712774397400479}. Best is trial 41 with value: 0.17867968268861573.
Fold 1 IBS: 0.16130446979847196
Fold 2 IBS: 0.17962788080780573
Fold 3 IBS: 0.15234271611359756
Fold 4 IBS: 0.14639704505503323
Fold 5 IBS: 0.2535383625662432
[I 2024-04-13 23:57:37,281] Trial 51 finished with value: 0.17864209486823032 and parameters: {'l1_ratio': 0.05123710958654414}. Best is trial 51 with value: 0.17864209486823032.
Fold 1 IBS: 0.16130838901537564
Fold 2 IBS: 0.17984096898703097
Fold 3 IBS: 0.15238003779253095
Fold 4 IBS: 0.14646021

Fold 1 IBS: 0.16131958581546293
Fold 2 IBS: 0.1798894318751477
Fold 3 IBS: 0.15237772825717347
Fold 4 IBS: 0.14642935548026983
Fold 5 IBS: 0.2536095439398616
[I 2024-04-13 23:58:38,407] Trial 74 finished with value: 0.1787251290735831 and parameters: {'l1_ratio': 0.10069635105041849}. Best is trial 51 with value: 0.17864209486823032.
Fold 1 IBS: 0.16131433492773814
Fold 2 IBS: 0.1797844922307758
Fold 3 IBS: 0.1523682154464997
Fold 4 IBS: 0.14642414274758808
Fold 5 IBS: 0.25359874998491827
[I 2024-04-13 23:58:41,552] Trial 75 finished with value: 0.178697987067504 and parameters: {'l1_ratio': 0.08329105235709114}. Best is trial 51 with value: 0.17864209486823032.
Fold 1 IBS: 0.16129911359955093
Fold 2 IBS: 0.179562830569666
Fold 3 IBS: 0.1523762695331006
Fold 4 IBS: 0.2225025243442987
Fold 5 IBS: 0.25365554767915277
[I 2024-04-13 23:58:44,959] Trial 76 finished with value: 0.1938792571451538 and parameters: {'l1_ratio': 0.028512034772785286}. Best is trial 51 with value: 0.1786420948682

Fold 5 IBS: 0.2535633964402299
[I 2024-04-14 00:00:04,170] Trial 98 finished with value: 0.17867102308294952 and parameters: {'l1_ratio': 0.0682397155252202}. Best is trial 51 with value: 0.17864209486823032.
Fold 1 IBS: 0.16130916156891112
Fold 2 IBS: 0.1797745136142011
Fold 3 IBS: 0.1523983224136218
Fold 4 IBS: 0.14649286610500428
Fold 5 IBS: 0.25370183595794904
[I 2024-04-14 00:00:08,541] Trial 99 finished with value: 0.1787353399319375 and parameters: {'l1_ratio': 0.0667304121180121}. Best is trial 51 with value: 0.17864209486823032.


* Best trial for IBS: 
 FrozenTrial(number=51, state=TrialState.COMPLETE, values=[0.17864209486823032], datetime_start=datetime.datetime(2024, 4, 13, 23, 57, 33, 815139), datetime_complete=datetime.datetime(2024, 4, 13, 23, 57, 37, 280204), params={'l1_ratio': 0.05123710958654414}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=51, value=None)

In [48]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.759
train_ibs:  0.179


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.4231641494784485)

test_cindex : 0.571


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.05123710958654414)

test_ibs:  0.275


In [52]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [85]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 12:49:38,775] A new study created in memory with name: no-name-f05b2836-6c34-4d37-ac66-b8eca987a59c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.609375
Fold 3 C-index: 0.7916666666666666
Fold 4 C-index: 0.8459915611814346
Fold 5 C-index: 0.636150234741784
[I 2024-04-14 12:49:42,443] Trial 0 finished with value: 0.7281518440331286 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7281518440331286.
Fold 1 C-index: 0.7683982683982684
Fold 2 C-index: 0.6517857142857143
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.6431924882629108
[I 2024-04-14 12:49:44,943] Trial 1 finished with value: 0.7457480169493687 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_featur

Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:50:28,034] Trial 15 finished with value: 0.5 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 2, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 51, 'oob_score': True, 'max_samples': 0.3692598016141903, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.19402726021030228, 'warm_start': True}. Best is trial 14 with value: 0.7600266365278019.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:50:30,650] Trial 16 finished with value: 0.5 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 17, 'min_samples_leaf': 10, 'max_depth': 1, 'n_estimators': 122, 'oob_score': True, 'max_samples': 0.5327960974085133, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.38557060567081813, 'warm_start': True}. Best is trial 14 with value: 0.7600266365278019.
Fold 1 C-index: 0.79004329004329
Fold 2 C-index: 0.7566964285714286
Fold 3 C-index: 0.8480392156

Fold 1 C-index: 0.7380952380952381
Fold 2 C-index: 0.7522321428571429
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8586497890295358
Fold 5 C-index: 0.6619718309859155
[I 2024-04-14 12:50:48,819] Trial 31 finished with value: 0.7659152903896449 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 7, 'min_samples_leaf': 9, 'max_depth': 13, 'n_estimators': 165, 'oob_score': True, 'max_samples': 0.25074023423629904, 'max_features': None, 'min_weight_fraction_leaf': 0.04816721016288783, 'warm_start': True}. Best is trial 17 with value: 0.7892309398880706.
Fold 1 C-index: 0.7380952380952381
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8649789029535865
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:50:49,498] Trial 32 finished with value: 0.7843718803263862 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 6, 'min_samples_leaf': 9, 'max_depth': 16, 'n_estimators': 77, 'oob_score': True, 'max_samples': 0.5438109085992663, '

...

Fold 1 IBS: 0.21398563901141657
Fold 2 IBS: 0.22134596740550985
Fold 3 IBS: 0.2048025990579434
Fold 4 IBS: 0.22461415055237854
Fold 5 IBS: 0.21851803782809134
[I 2024-04-14 12:54:15,787] Trial 32 finished with value: 0.21665327877106794 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 451, 'oob_score': False, 'max_samples': 0.7396444271446992, 'max_features': None, 'min_weight_fraction_leaf': 0.4501495537267681}. Best is trial 23 with value: 0.18256840611756237.
Fold 1 IBS: 0.1881712152839018
Fold 2 IBS: 0.19602908273271186
Fold 3 IBS: 0.187050921824735
Fold 4 IBS: 0.18923643256610323
Fold 5 IBS: 0.21856358775020004
[I 2024-04-14 12:54:20,361] Trial 33 finished with value: 0.19581024803153038 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 12, 'min_samples_leaf': 1, 'max_depth': 19, 'n_estimators': 493, 'oob_score': False, 'max_samples': 0.9933327757837453, 'max_features': None, 'min_weight_fraction_leaf'

Fold 1 IBS: 0.1705674173139149
Fold 2 IBS: 0.1852617720416809
Fold 3 IBS: 0.16929920721077502
Fold 4 IBS: 0.16857228614310335
Fold 5 IBS: 0.23285081643644664
[I 2024-04-14 12:55:17,808] Trial 48 finished with value: 0.18531029982918418 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 13, 'min_samples_leaf': 5, 'max_depth': 15, 'n_estimators': 431, 'oob_score': False, 'max_samples': 0.9996528707471266, 'max_features': None, 'min_weight_fraction_leaf': 0.47380170342998185}. Best is trial 23 with value: 0.18256840611756237.
Fold 1 IBS: 0.21397420828984268
Fold 2 IBS: 0.22140730914697682
Fold 3 IBS: 0.2048165899556378
Fold 4 IBS: 0.22466454285026752
Fold 5 IBS: 0.2185528446348627
[I 2024-04-14 12:55:20,811] Trial 49 finished with value: 0.21668309897551746 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 8, 'min_samples_leaf': 20, 'max_depth': 6, 'n_estimators': 330, 'oob_score': False, 'max_samples': 0.8459565404966409, 'max_features': 'auto', 'min_weight_fraction_lea

Fold 1 IBS: 0.17758210577267103
Fold 2 IBS: 0.18839515678161117
Fold 3 IBS: 0.17378027525748194
Fold 4 IBS: 0.17402263926693903
Fold 5 IBS: 0.23329928868140723
[I 2024-04-14 12:56:19,002] Trial 64 finished with value: 0.1894158931520221 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 20, 'n_estimators': 421, 'oob_score': False, 'max_samples': 0.8662053800926933, 'max_features': None, 'min_weight_fraction_leaf': 0.4182804957604697}. Best is trial 54 with value: 0.1822436647018153.
Fold 1 IBS: 0.17735779335885185
Fold 2 IBS: 0.18791831745948914
Fold 3 IBS: 0.17437629608751362
Fold 4 IBS: 0.1754060616546171
Fold 5 IBS: 0.2347254760738243
[I 2024-04-14 12:56:22,530] Trial 65 finished with value: 0.18995678892685922 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 12, 'min_samples_leaf': 7, 'max_depth': 18, 'n_estimators': 380, 'oob_score': False, 'max_samples': 0.9168825987611354, 'max_features': None, 'min_weight_fraction_leaf':

Fold 1 IBS: 0.1754183266894564
Fold 2 IBS: 0.2119163985327653
Fold 3 IBS: 0.16611082563106694
Fold 4 IBS: 0.16255778819587066
Fold 5 IBS: 0.22916546086072462
[I 2024-04-14 12:57:17,485] Trial 80 finished with value: 0.18903375998197677 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 3, 'max_depth': 12, 'n_estimators': 237, 'oob_score': False, 'max_samples': 0.9992480396864105, 'max_features': None, 'min_weight_fraction_leaf': 0.40510470687091726}. Best is trial 54 with value: 0.1822436647018153.
Fold 1 IBS: 0.16993730570122387
Fold 2 IBS: 0.195387539069006
Fold 3 IBS: 0.1651714087990093
Fold 4 IBS: 0.16201193892631496
Fold 5 IBS: 0.22567140464684687
[I 2024-04-14 12:57:20,276] Trial 81 finished with value: 0.18363591942848018 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 16, 'min_samples_leaf': 7, 'max_depth': 15, 'n_estimators': 296, 'oob_score': False, 'max_samples': 0.8729504953795615, 'max_features': None, 'min_weight_fraction_leaf': 0

Fold 1 IBS: 0.17691142196188117
Fold 2 IBS: 0.1884356638717461
Fold 3 IBS: 0.17473493092732248
Fold 4 IBS: 0.1743581409278057
Fold 5 IBS: 0.23455157057349743
[I 2024-04-14 12:58:13,521] Trial 96 finished with value: 0.18979834565245057 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 15, 'min_samples_leaf': 6, 'max_depth': 2, 'n_estimators': 412, 'oob_score': False, 'max_samples': 0.9548184208743441, 'max_features': None, 'min_weight_fraction_leaf': 0.46352447817689174}. Best is trial 94 with value: 0.18218245586616172.
Fold 1 IBS: 0.1720334166380091
Fold 2 IBS: 0.19310219127184633
Fold 3 IBS: 0.16467180888297991
Fold 4 IBS: 0.16215325103272105
Fold 5 IBS: 0.225731258677061
[I 2024-04-14 12:58:17,676] Trial 97 finished with value: 0.1835383853005235 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 14, 'min_samples_leaf': 8, 'max_depth': 2, 'n_estimators': 434, 'oob_score': False, 'max_samples': 0.9862595341313025, 'max_features': None, 'min_weight_fraction_leaf': 0.

In [86]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [87]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.855
train_ibs:  0.182


#### Test

In [88]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [89]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=14, max_features=None, max_leaf_nodes=11,
                     max_samples=0.7127178148051858, min_samples_leaf=1,
                     min_samples_split=8,
                     min_weight_fraction_leaf=0.01586498778855871,
                     n_estimators=236, random_state=123, warm_start=True)

test_cindex:  0.643


RandomSurvivalForest(max_depth=2, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9526180318811873, min_samples_leaf=6,
                     min_samples_split=7,
                     min_weight_fraction_leaf=0.4336258803047899,
                     n_estimators=408, random_state=123)

test_ibs:  0.206


In [90]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [91]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [92]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 12:58:26,868] A new study created in memory with name: no-name-9a32973d-e195-4471-a6f4-ed8e972ebd65


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7640692640692641
Fold 2 C-index: 0.7589285714285714
Fold 3 C-index: 0.8431372549019608
Fold 4 C-index: 0.8354430379746836
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:58:27,792] Trial 0 finished with value: 0.7745879261443795 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7745879261443795.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 12:58:30,316] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7467532467532467
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8565400843881856
Fold 5 C-index: 0.6549295774647887
[I 2024-04-14 12:58:45,399] Trial 16 finished with value: 0.7829471027296476 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 237, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.8988170687511211, 'min_weight_fraction_leaf': 0.1144909881867055}. Best is trial 16 with value: 0.7829471027296476.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.8627450980392157
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.6666666666666666
[I 2024-04-14 12:58:45,885] Trial 17 finished with value: 0.7903075269233125 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 6, 'max_depth': 17, 'n_estimators': 256, 'oob_score': False, 'warm_start': True, 'max_feature

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.8676470588235294
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6713615023474179
[I 2024-04-14 12:58:54,064] Trial 31 finished with value: 0.789727434191557 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 17, 'n_estimators': 235, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.9117101001226512, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 25 with value: 0.8076326900231233.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.8214285714285714
Fold 3 C-index: 0.8774509803921569
Fold 4 C-index: 0.8481012658227848
Fold 5 C-index: 0.6948356807511737
[I 2024-04-14 12:58:54,648] Trial 32 finished with value: 0.7998784511940888 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 13, 'n_estimators': 236, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_

...

Fold 1 IBS: 0.16288193492716865
Fold 2 IBS: 0.2145204309047712
Fold 3 IBS: 0.16344681522379592
Fold 4 IBS: 0.1607591766490642
Fold 5 IBS: 0.23285217315239765
[I 2024-04-14 13:00:55,322] Trial 30 finished with value: 0.18689210617143953 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 19, 'min_samples_leaf': 4, 'max_depth': 7, 'n_estimators': 87, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.9504933061433887, 'min_weight_fraction_leaf': 0.1352883886134873}. Best is trial 7 with value: 0.18575431268869705.
Fold 1 IBS: 0.1522561822367273
Fold 2 IBS: 0.22165813527341477
Fold 3 IBS: 0.15806345503003275
Fold 4 IBS: 0.14643989125013568
Fold 5 IBS: 0.23525263253654355
[I 2024-04-14 13:00:57,109] Trial 31 finished with value: 0.18273405926537079 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 197, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.

Fold 1 IBS: 0.154877427493089
Fold 2 IBS: 0.2119381972371029
Fold 3 IBS: 0.16230571380133155
Fold 4 IBS: 0.15585982768129883
Fold 5 IBS: 0.23020111518690609
[I 2024-04-14 13:01:17,613] Trial 45 finished with value: 0.18303645627994566 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 2, 'max_depth': 5, 'n_estimators': 139, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.7784599232837964, 'min_weight_fraction_leaf': 0.0038388411371728937}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.1807341100446867
Fold 2 IBS: 0.20797545001135234
Fold 3 IBS: 0.17776524098244717
Fold 4 IBS: 0.18082034578648548
Fold 5 IBS: 0.22062001984045795
[I 2024-04-14 13:01:18,767] Trial 46 finished with value: 0.19358303333308596 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 7, 'min_samples_leaf': 12, 'max_depth': 3, 'n_estimators': 138, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.783

Fold 1 IBS: 0.1667377112785077
Fold 2 IBS: 0.21139971336963634
Fold 3 IBS: 0.16719610422116873
Fold 4 IBS: 0.16444742789928804
Fold 5 IBS: 0.22433929414894066
[I 2024-04-14 13:01:43,336] Trial 60 finished with value: 0.18682405018350828 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 7, 'min_samples_leaf': 6, 'max_depth': 5, 'n_estimators': 233, 'oob_score': False, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.9208992051748564, 'min_weight_fraction_leaf': 0.04194475489343627}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.15339272842252286
Fold 2 IBS: 0.22875929926363106
Fold 3 IBS: 0.15824814328892167
Fold 4 IBS: 0.14551047397439773
Fold 5 IBS: 0.23751514003516505
[I 2024-04-14 13:01:44,750] Trial 61 finished with value: 0.1846851569969277 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 16, 'min_samples_leaf': 2, 'max_depth': 9, 'n_estimators': 182, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.8

Fold 1 IBS: 0.15528894958073775
Fold 2 IBS: 0.21984680247745453
Fold 3 IBS: 0.15900685739714732
Fold 4 IBS: 0.14859781459794244
Fold 5 IBS: 0.23493449886479167
[I 2024-04-14 13:02:08,119] Trial 75 finished with value: 0.18353498458361475 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 162, 'oob_score': True, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.9532062735167633, 'min_weight_fraction_leaf': 0.030316433196080697}. Best is trial 31 with value: 0.18273405926537079.
Fold 1 IBS: 0.1635808951920651
Fold 2 IBS: 0.20759879371938625
Fold 3 IBS: 0.16635458220500324
Fold 4 IBS: 0.1603635273963648
Fold 5 IBS: 0.2261410117149779
[I 2024-04-14 13:02:09,467] Trial 76 finished with value: 0.18480776204555946 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 5, 'n_estimators': 128, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.96

Fold 2 IBS: 0.2063565222747697
Fold 3 IBS: 0.17003588715814522
Fold 4 IBS: 0.17777116278712887
Fold 5 IBS: 0.22022528125047466
[I 2024-04-14 13:02:19,318] Trial 90 finished with value: 0.18995479834281107 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 4, 'n_estimators': 37, 'oob_score': True, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.9997596356740963, 'min_weight_fraction_leaf': 0.3292134206382089}. Best is trial 81 with value: 0.1804545979151259.
Fold 1 IBS: 0.15458861367684845
Fold 2 IBS: 0.21750786042177303
Fold 3 IBS: 0.15767300132028467
Fold 4 IBS: 0.15084191777051614
Fold 5 IBS: 0.23638504482176886
[I 2024-04-14 13:02:20,044] Trial 91 finished with value: 0.18339928760223825 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 18, 'min_samples_leaf': 2, 'max_depth': 6, 'n_estimators': 60, 'oob_score': True, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.916420719730876, 'min_weight_fract

In [93]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [94]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.848
train_ibs:  0.18


#### Test

In [95]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [96]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=10, max_features=None, max_leaf_nodes=17,
                   max_samples=0.7657259832611387, min_samples_leaf=1,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.007259167100140962,
                   n_estimators=358, random_state=123, warm_start=True)

C-index score: 0.61


ExtraSurvivalTrees(max_depth=5, max_leaf_nodes=17,
                   max_samples=0.9277129016361768, min_samples_leaf=1,
                   min_samples_split=16,
                   min_weight_fraction_leaf=0.026154087319717897,
                   n_estimators=27, oob_score=True, random_state=123)

IBS: 0.23


In [97]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [98]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-14 13:02:24,950] A new study created in memory with name: no-name-634d57e3-44db-4212-9920-d8b077810fed


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:02:39,856] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 13:02:47,293] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:28:31,503] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7308620854904466.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:30:08,794] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-14 14:39:54,383] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7308620854904466.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6197183098591549
[I 2024-04-14 14:40:32,175] Trial 26 finished with value: 0.5440329476861168 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446,

...

Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:27:07,942] Trial 55 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.9027683411994925, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.13472529918097312, 'n_estimators': 381, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.36938177041618503, 'max_features': 'auto', 'min_impurity_decrease': 1.1016843774656315e-07, 'validation_fraction': 0.898549324711475, 'min_samples_split': 3, 'max_leaf_nodes': 12, 'min_samples_leaf': 12, 'max_depth': 3}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:27:42,167] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.83247895380525

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:35:59,995] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9998769156045215, 'learning_rate': 0.008782667792366316, 'dropout_rate': 0.12972298735589705, 'n_estimators': 500, 'criterion': 'squared_error', 'ccp_alpha': 0.7134499429690735, 'min_weight_fraction_leaf': 0.19311178079767077, 'max_features': 'log2', 'min_impurity_decrease': 1.142816958470248e-06, 'validation_fraction': 0.9729459657934212, 'min_samples_split': 18, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 4}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 15:36:51,016] Trial 67 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.864018533057457

Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:43:26,046] Trial 77 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527746876723795, 'learning_rate': 0.05776064499915696, 'dropout_rate': 0.2556920821676466, 'n_estimators': 442, 'criterion': 'squared_error', 'ccp_alpha': 0.5896250392603071, 'min_weight_fraction_leaf': 0.22988050749684658, 'max_features': 1, 'min_impurity_decrease': 1.4995945809430595e-05, 'validation_fraction': 0.8950354974332698, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 11, 'max_depth': 3}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:44:15,351] Trial 78 finished with value: 0.21659054862241586 and parameters: {'su

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-14 15:51:18,817] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8800231052433968, 'learning_rate': 0.015424283294721588, 'dropout_rate': 0.20093805843087115, 'n_estimators': 390, 'criterion': 'squared_error', 'ccp_alpha': 1.351539840282731, 'min_weight_fraction_leaf': 0.28958635214901884, 'max_features': 'auto', 'min_impurity_decrease': 2.329078771156988e-07, 'validation_fraction': 0.939344740485312, 'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 4}. Best is trial 53 with value: 0.21504716009286912.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 15:51:49,189] Trial 89 finished with value: 0.21659054862241586 and parameters: {'

Fold 1 IBS: 0.21252354672208404
Fold 2 IBS: 0.22052727078684145
Fold 3 IBS: 0.20320794231873263
Fold 4 IBS: 0.2228335466693306
Fold 5 IBS: 0.21782914318597227
[I 2024-04-14 16:00:27,913] Trial 99 finished with value: 0.21538428993659223 and parameters: {'subsample': 0.24286963781937687, 'learning_rate': 0.01358129947690588, 'dropout_rate': 0.25496540816023416, 'n_estimators': 454, 'criterion': 'squared_error', 'ccp_alpha': 0.004367160733265784, 'min_weight_fraction_leaf': 0.2624099419664444, 'max_features': 'auto', 'min_impurity_decrease': 5.321953542697131e-07, 'validation_fraction': 0.9620562032012763, 'min_samples_split': 11, 'max_leaf_nodes': 20, 'min_samples_leaf': 10, 'max_depth': 1}. Best is trial 53 with value: 0.21504716009286912.


* Best trial for IBS: 
 FrozenTrial(number=53, state=TrialState.COMPLETE, values=[0.21504716009286912], datetime_start=datetime.datetime(2024, 4, 14, 15, 25, 6, 566505), datetime_complete=datetime.datetime(2024, 4, 14, 15, 25, 57, 272952), params={

In [99]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [100]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.74
train_ibs:  0.215


#### Test

In [101]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [102]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.03063070291051248,
                                 criterion='squared_error',
                                 dropout_rate=0.2576884847115747,
                                 learning_rate=0.007359366951045268,
                                 max_depth=16, max_features=1,
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=1.3903906488794697e-07,
                                 min_samples_leaf=14, min_samples_split=20,
                                 min_weight_fraction_leaf=0.38180626899357545,
                                 n_estimators=96, random_state=123,
                                 subsample=0.8938290428827321,
                                 validation_fraction=0.9577535215137098)

C-index score: 0.612


GradientBoostingSurvivalAnalysis(ccp_alpha=0.009625013743012712,
                                 criterion='squared_error',
                                 dropout_rate=0.1897783294507234,
                                 learning_rate=0.015420772490455037,
                                 max_features='auto', max_leaf_nodes=14,
                                 min_impurity_decrease=5.869825897765074e-07,
                                 min_samples_leaf=13, min_samples_split=20,
                                 min_weight_fraction_leaf=0.29880213170319914,
                                 n_estimators=430, random_state=123,
                                 subsample=0.9101431135071837,
                                 validation_fraction=0.9964423942006735)

IBS: 0.22


In [103]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [104]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [105]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 16:00:40,064] A new study created in memory with name: no-name-eceb31d0-e282-45c5-8cd5-4b90d8246572


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6339285714285714
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.5938967136150235
[I 2024-04-14 16:00:41,024] Trial 0 finished with value: 0.7099982410993864 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.7099982410993864.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.625
Fold 3 C-index: 0.7450980392156863
Fold 4 C-index: 0.7848101265822784
Fold 5 C-index: 0.5985915492957746
[I 2024-04-14 16:00:47,475] Trial 1 finished with value: 0.6987518910706959 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.7099982410993864.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.6294642857142857
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.

Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.6651785714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8396624472573839
Fold 5 C-index: 0.6150234741784038
[I 2024-04-14 16:01:42,766] Trial 19 finished with value: 0.7380039655451153 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9936895487898989, 'n_estimators': 431, 'learning_rate': 0.07862475807967083}. Best is trial 19 with value: 0.7380039655451153.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.625
Fold 3 C-index: 0.7843137254901961
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.5821596244131455
[I 2024-04-14 16:01:46,275] Trial 20 finished with value: 0.7169203481000173 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.7933084651006226, 'n_estimators': 401, 'learning_rate': 0.06551866052750378}. Best is trial 19 with value: 0.7380039655451153.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.6607142857142857
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index

Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.6651785714285714
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8396624472573839
Fold 5 C-index: 0.6103286384976526
[I 2024-04-14 16:02:53,883] Trial 38 finished with value: 0.7379307992747658 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.4920103977585895, 'n_estimators': 342, 'learning_rate': 0.0787855279203912}. Best is trial 19 with value: 0.7380039655451153.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.6473214285714286
Fold 3 C-index: 0.7769607843137255
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.5915492957746479
[I 2024-04-14 16:02:57,111] Trial 39 finished with value: 0.7217919798513094 and parameters: {'subsample': 0.22913341588947253, 'dropout_rate': 0.48327859647662685, 'n_estimators': 344, 'learning_rate': 0.07135359751067862}. Best is trial 19 with value: 0.7380039655451153.
Fold 1 C-index: 0.7662337662337663
Fold 2 C-index: 0.65625
Fold 3 C-index: 0.7941176470588235
Fold 4 C-in

...

Fold 2 IBS: 0.2354300841917142
Fold 3 IBS: 0.16162421990807277
Fold 4 IBS: 0.21881379377656782
Fold 5 IBS: 0.2684022959740717
[I 2024-04-14 16:07:25,516] Trial 19 finished with value: 0.2105620453048207 and parameters: {'subsample': 0.16667191735421055, 'dropout_rate': 0.995336953668152, 'n_estimators': 400, 'learning_rate': 0.0164716167796114}. Best is trial 16 with value: 0.1937517153099759.
Fold 1 IBS: 0.23230314629982232
Fold 2 IBS: 0.3472907284410055
Fold 3 IBS: 0.24260871569057887
Fold 4 IBS: 0.3158666317843052
Fold 5 IBS: 0.3305681079680327
[I 2024-04-14 16:07:27,147] Trial 20 finished with value: 0.29372746603674893 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.7933084651006226, 'n_estimators': 307, 'learning_rate': 0.0817175642960023}. Best is trial 16 with value: 0.1937517153099759.
Fold 1 IBS: 0.1867095271065796
Fold 2 IBS: 0.1938189738975239
Fold 3 IBS: 0.17171699702984686
Fold 4 IBS: 0.19765663176182818
Fold 5 IBS: 0.2217748556695782
[I 2024-04-14 16

Fold 3 IBS: 0.1644847044499737
Fold 4 IBS: 0.21105160502473713
Fold 5 IBS: 0.25612451029896155
[I 2024-04-14 16:07:45,482] Trial 38 finished with value: 0.206199007987175 and parameters: {'subsample': 0.23557094034920928, 'dropout_rate': 0.9772185725180188, 'n_estimators': 170, 'learning_rate': 0.028634921614009247}. Best is trial 32 with value: 0.193435014475276.
Fold 1 IBS: 0.18527875729076687
Fold 2 IBS: 0.206842479595501
Fold 3 IBS: 0.1684391799542552
Fold 4 IBS: 0.20310310442935434
Fold 5 IBS: 0.23670810681271148
[I 2024-04-14 16:07:46,594] Trial 39 finished with value: 0.20007432561651778 and parameters: {'subsample': 0.34419137871914335, 'dropout_rate': 0.7641491998417398, 'n_estimators': 203, 'learning_rate': 0.013931445007643331}. Best is trial 32 with value: 0.193435014475276.
Fold 1 IBS: 0.17416970091294498
Fold 2 IBS: 0.2527135803516107
Fold 3 IBS: 0.18506003766329993
Fold 4 IBS: 0.24375993453341516
Fold 5 IBS: 0.2807864950347834
[I 2024-04-14 16:07:48,370] Trial 40 finishe

Fold 1 IBS: 0.17625443541490904
Fold 2 IBS: 0.21448734172456063
Fold 3 IBS: 0.159694803303589
Fold 4 IBS: 0.19917983886935933
Fold 5 IBS: 0.24753611571578277
[I 2024-04-14 16:07:56,077] Trial 58 finished with value: 0.19943050700564013 and parameters: {'subsample': 0.2868686808366677, 'dropout_rate': 0.2032153865804448, 'n_estimators': 65, 'learning_rate': 0.06902571928702043}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.20677608349073381
Fold 2 IBS: 0.2755720028953149
Fold 3 IBS: 0.2188752151078084
Fold 4 IBS: 0.3174407719124043
Fold 5 IBS: 0.3329212341311742
[I 2024-04-14 16:07:58,586] Trial 59 finished with value: 0.27031706150748713 and parameters: {'subsample': 0.1311152882733824, 'dropout_rate': 0.1402923749214668, 'n_estimators': 308, 'learning_rate': 0.0868909336659619}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.1749843583986033
Fold 2 IBS: 0.2596945560423311
Fold 3 IBS: 0.17459495798534216
Fold 4 IBS: 0.27092585930374996
Fold 5 IBS: 0.

Fold 1 IBS: 0.17659749400643285
Fold 2 IBS: 0.20287239360600032
Fold 3 IBS: 0.15841071050681252
Fold 4 IBS: 0.18458264033581556
Fold 5 IBS: 0.23899965212870403
[I 2024-04-14 16:08:04,445] Trial 77 finished with value: 0.19229257811675307 and parameters: {'subsample': 0.20832899863841498, 'dropout_rate': 0.14156682139259733, 'n_estimators': 51, 'learning_rate': 0.08592392402640006}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.1636303349328578
Fold 2 IBS: 0.22857784006418638
Fold 3 IBS: 0.1605793070639578
Fold 4 IBS: 0.18309179597728542
Fold 5 IBS: 0.2739548394022102
[I 2024-04-14 16:08:05,101] Trial 78 finished with value: 0.2019668234880995 and parameters: {'subsample': 0.12123242375860951, 'dropout_rate': 0.22328006044976534, 'n_estimators': 117, 'learning_rate': 0.07196105986543508}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.16739080446957846
Fold 2 IBS: 0.22120867741229525
Fold 3 IBS: 0.1564177245718657
Fold 4 IBS: 0.19727210170171616
Fold 5

Fold 2 IBS: 0.2124197904403002
Fold 3 IBS: 0.19452025866502365
Fold 4 IBS: 0.21624395545790184
Fold 5 IBS: 0.21727988373559398
[I 2024-04-14 16:08:21,350] Trial 96 finished with value: 0.20883327344060998 and parameters: {'subsample': 0.14027395841703735, 'dropout_rate': 0.121620990717481, 'n_estimators': 14, 'learning_rate': 0.0457852425641791}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.19746651082582445
Fold 2 IBS: 0.20323222090294873
Fold 3 IBS: 0.1846720080714762
Fold 4 IBS: 0.21031490576039452
Fold 5 IBS: 0.2178337569161985
[I 2024-04-14 16:08:21,536] Trial 97 finished with value: 0.20270388049536853 and parameters: {'subsample': 0.16905792666412153, 'dropout_rate': 0.1895528405665567, 'n_estimators': 26, 'learning_rate': 0.049285110803541435}. Best is trial 53 with value: 0.18917045101225236.
Fold 1 IBS: 0.1762371567159911
Fold 2 IBS: 0.24559502511753464
Fold 3 IBS: 0.1641582028784399
Fold 4 IBS: 0.2375544451681581
Fold 5 IBS: 0.2722735472612508
[I 2024-04-1

In [106]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [107]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.74
train_ibs:  0.189


#### Test

In [108]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [109]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.17771784288698328,
                                              learning_rate=0.06006835823961436,
                                              n_estimators=396,
                                              random_state=123,
                                              subsample=0.1001094180478445)

C-index score: 0.53


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10188563743505225,
                                              learning_rate=0.07845638174276695,
                                              n_estimators=68, random_state=123,
                                              subsample=0.18457062310520533)

IBS: 0.256


In [110]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [111]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.855,1.0
ExtraSurvivalTrees,0.848,2.0
CoxPH,0.761,3.0
CoxElastic,0.759,4.0
CoxLasso,0.758,5.0
GradientBoosting,0.740,6.5
ComponentwiseGradientBoosting,0.740,6.5
CoxRidge,0.691,8.0


In [112]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxPH,0.179,1.5
CoxElastic,0.179,1.5
CoxLasso,0.180,3.5
ExtraSurvivalTrees,0.180,3.5
Randomsurvivalforest,0.182,5.0
ComponentwiseGradientBoosting,0.189,6.0
GradientBoosting,0.215,7.0
CoxRidge,0.217,8.0


In [113]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
Randomsurvivalforest,0.643,1.0
GradientBoosting,0.612,2.0
ExtraSurvivalTrees,0.610,3.0
CoxLasso,0.571,4.5
CoxElastic,0.571,4.5
CoxPH,0.568,6.0
CoxRidge,0.534,7.0
ComponentwiseGradientBoosting,0.530,8.0


In [114]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
Randomsurvivalforest,0.206,1.0
GradientBoosting,0.220,2.0
CoxRidge,0.221,3.0
ExtraSurvivalTrees,0.230,4.0
ComponentwiseGradientBoosting,0.256,5.0
CoxLasso,0.275,6.5
CoxElastic,0.275,6.5
CoxPH,0.279,8.0


In [115]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = 'path_to_your_folder/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  # List of your DataFrames
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/standard/rent/'  # Folder path where you want to save the files

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

modified_file_names = ['d1_os_standard_rent_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [116]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
